In [ ]:
import pandas as pd
import numpy as np

In [ ]:
predictions = pd.read_csv('./rnadegformer_extras/best_pl_weights_subs/submission.csv')

In [ ]:
test_set = pd.read_json('./RNAdegformer/src/OpenVaccine/data/test_with_labels.json', lines=True)

In [ ]:
predictions['id']  = predictions['id_seqpos'].str.split('_').apply(lambda x: x[0]+ '_' +x[1])

In [ ]:
id = []
ph10 = []
Mg50 = []
reactivity = []
for i in np.unique(predictions['id']):
    filtered_ph10 = list(predictions[predictions['id'] == i]['deg_pH10'])
    filtered_50C = list(predictions[predictions['id'] == i]['deg_Mg_50C'])
    filtered_reactivity = list(predictions[predictions['id'] == i]['reactivity'])
    id.append(i)
    ph10.append(filtered_ph10)
    Mg50.append(filtered_50C)
    reactivity.append(filtered_reactivity)

In [ ]:
predictions_df = pd.DataFrame({'id': id, 'predicted_deg_ph10': ph10, 'predicted_deg_Mg_50C': Mg50, 'predicted_reactivity': reactivity})

In [ ]:
merged_results = predictions_df.merge(test_set[['id', 'deg_pH10', 'deg_Mg_50C', 'reactivity']], left_on='id', right_on='id')

In [ ]:
merged_results.rename(columns={'deg_pH10': 'actual_deg_pH10', 'deg_Mg_50C': 'actual_deg_Mg_50C', 'reactivity': 'actual_reactivity'}, inplace=True)

In [ ]:
actual_degph10 = merged_results['actual_deg_pH10'].apply(lambda x: x[:68]).to_numpy()
actual_deg50Mg = merged_results['actual_deg_Mg_50C'].apply(lambda x: x[:68]).to_numpy()
actual_reactivity = merged_results['actual_reactivity'].apply(lambda x: x[:68]).to_numpy()

predicted_deg_ph10 = merged_results['predicted_deg_ph10'].apply(lambda x: x[:68]).to_numpy()
predicted_deg50Mg = merged_results['predicted_deg_Mg_50C'].apply(lambda x: x[:68]).to_numpy()
predicted_reactivity = merged_results['predicted_reactivity'].apply(lambda x: x[:68]).to_numpy()

In [ ]:
actual_flat = np.concatenate([np.array(x) for x in actual_degph10])
predicted_flat = np.concatenate([np.array(x) for x in predicted_deg_ph10])

ph10_mcrmse = np.sqrt(np.mean((actual_flat - predicted_flat) ** 2))

print(ph10_mcrmse)

In [ ]:
actual_flat = np.concatenate([np.array(x) for x in actual_deg50Mg])
predicted_flat = np.concatenate([np.array(x) for x in predicted_deg50Mg])

Mg50C_mcrmse = np.sqrt(np.mean((actual_flat - predicted_flat) ** 2))

print(Mg50C_mcrmse)

In [ ]:
actual_flat = np.concatenate([np.array(x) for x in actual_reactivity])
predicted_flat = np.concatenate([np.array(x) for x in predicted_reactivity])

reactivity_mcrmse = np.sqrt(np.mean((actual_flat - predicted_flat) ** 2))

print(reactivity_mcrmse)

In [ ]:
mcrmse = np.mean([ph10_mcrmse, Mg50C_mcrmse, reactivity_mcrmse])

In [ ]:
print(mcrmse)